# SFT: дообучение Qwen3.5-0.8B для русской орфографии

Supervised Fine-Tuning на датасете [`BW/RU_SPELLCHECK_DEVICE`](https://huggingface.co/datasets/BW/RU_SPELLCHECK_DEVICE) с помощью [Unsloth](https://unsloth.ai/) в Google Colab (GPU T4).

**Runtime → Run all**

## 1. Авторизация Hugging Face

In [ ]:
import os
from huggingface_hub import login

# Colab: можно использовать секрет HF_TOKEN
# from google.colab import userdata
# hf_token = userdata.get('HF_TOKEN')

hf_token = os.environ.get('HF_TOKEN', '')
login(hf_token)


## 2. Установка зависимостей

In [ ]:
%%capture
import os, re

if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch
    v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"

!pip install --no-deps transformers==5.5.0 "tokenizers>=0.22.0,<=0.23.0"
!pip install torchcodec

import torch
torch._dynamo.config.recompile_limit = 64


## 3. Загрузка базовой модели

In [ ]:
from unsloth import FastModel

MODEL_NAME = "unsloth/Qwen3.5-0.8B"
MAX_SEQ_LENGTH = 1024

model, tokenizer = FastModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=False,
    full_finetuning=False,
)


## 4. LoRA-адаптеры

In [ ]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers=False,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=8,
    lora_alpha=8,
    lora_dropout=0,
    bias="none",
    random_state=3407,
)


## 5. Chat template

In [ ]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template="qwen3-instruct",
)


## 6. Загрузка и подготовка данных

In [ ]:
from datasets import load_dataset

DATASET_NAME = "BW/RU_SPELLCHECK_DEVICE"
dataset = load_dataset(DATASET_NAME)
dataset


In [ ]:
def formatting_prompts_func(examples):
    texts = []
    for source_text, correction_text in zip(examples["source"], examples["correction"]):
        messages = [
            {"role": "user", "content": f"Исправь ошибки в тексте: {source_text}"},
            {"role": "assistant", "content": correction_text},
        ]
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )
        texts.append(text)
    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched=True)


## 7. Обучение

In [ ]:
from datasets import Dataset as HFDataset
from trl import SFTTrainer, SFTConfig
import torch

OUTPUT_DIR = "qwen-spellcheck-lora"
hf_dataset = HFDataset.from_list(dataset["train"])

trainer = SFTTrainer(
    model=model,
    train_dataset=hf_dataset,
    tokenizer=tokenizer,
    args=SFTConfig(
        output_dir=OUTPUT_DIR,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        max_steps=100,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        packing=False,
        max_seq_length=MAX_SEQ_LENGTH,
    ),
)


In [ ]:
trainer.train()


## 8. Сохранение и публикация на Hugging Face

In [ ]:
from huggingface_hub import HfApi

HF_REPO = "BW/Qwen3.5_Fine-tuned_on_RU_SPELLCHECK_DEVICE"

model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

api = HfApi(token=hf_token)
api.create_repo(repo_id=HF_REPO, repo_type="model", exist_ok=True)
model.push_to_hub(HF_REPO, token=hf_token)
tokenizer.push_to_hub(HF_REPO, token=hf_token)

print(f"Модель загружена: https://huggingface.co/{HF_REPO}")


## 9. Проверка инференса

In [ ]:
from transformers import TextStreamer

messages = [
    {"role": "user", "content": "Исправь ошибки в тексте: Првет!"},
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

inputs = tokenizer(
    text=[text],
    images=None,
    videos=None,
    return_tensors="pt",
).to(model.device)

_ = model.generate(
    **inputs,
    max_new_tokens=64,
    temperature=1.0,
    top_p=0.95,
    top_k=64,
    streamer=TextStreamer(tokenizer, skip_prompt=True),
)
